In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Tuple, Dict

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [ ]:
BASE = "/content/drive/MyDrive/Sophomore Year/IML_Fall2025_SkillVersusLuck"
IN_FP  = f"{BASE}/bundesliga_std.csv"
OUT_FP = f"{BASE}/Simulated Leagues/bundesliga_simulated_matches.csv"

df = pd.read_csv(IN_FP, dtype=str, low_memory=False)

In [33]:
need = ["season", "date", "home_team", "away_team"]
if any(c is None for c in need):
    raise ValueError(f"Missing expected columns. Got: {list(df.columns)[:15]}")

In [34]:
season_col = "season"
date_col = "date"
home_col = "home_team"
away_col = "away_team"
hg_col = "hometeamgoals"
ag_col = "awayteamgoals"
res_col = "hometeamresult"

ren = {
    season_col: "season",
    date_col:   "date",
    home_col:   "team1",
    away_col:   "team2",
}

if hg_col: ren[hg_col] = "team1_goals"
if ag_col: ren[ag_col] = "team2_goals"
if res_col: ren[res_col] = "true_home_team_result"

df = df.rename(columns=ren)

In [35]:
rng = np.random.default_rng(seed=42)

df["simulated_home_team_result"] = rng.choice([1, 0, -1], size=len(df), p=[1/3, 1/3, 1/3])

In [36]:
out_cols = ["season", "date", "team1", "team2", "true_home_team_result", "simulated_home_team_result"]

out = df[[c for c in out_cols if c in df.columns]].copy()

In [37]:
Path(BASE).mkdir(parents=True, exist_ok=True)
out.to_csv(OUT_FP, index=False)
print("Wrote:", OUT_FP)

Wrote: /content/drive/MyDrive/Sophomore Year/IML_Fall2025_SkillVersusLuck/Simulated Leagues/bundesliga_simulated_matches.csv


In [38]:
print("\nDistribution of simulated results (overall):")
print(out["simulated_home_team_result"].value_counts().sort_index())


Distribution of simulated results (overall):
simulated_home_team_result
-1    2080
 0    2122
 1    2224
Name: count, dtype: int64


In [39]:
print("\nFirst 10 rows:")
display(out.head(10))


First 10 rows:


,season,date,team1,team2,true_home_team_result,simulated_home_team_result
0,2024,2025-05-17,AUG,UNI,-1,-1
1,2024,2025-05-17,DOR,HOLK,1,0
2,2024,2025-05-17,FRE,EINF,-1,-1
3,2024,2025-05-17,HEI,WERB,-1,-1
4,2024,2025-05-17,HOF,BAY,-1,1
5,2024,2025-05-17,MAI,LEVK,0,-1
6,2024,2025-05-17,M'G,WOB,-1,-1
7,2024,2025-05-17,RBL,STU,-1,-1
8,2024,2025-05-17,STP,BOC,-1,1
9,2024,2025-05-11,LEVK,DOR,-1,0


In [8]:
from pathlib import Path
import numpy as np
import pandas as pd

In [9]:
HERE = Path.cwd()
LEAGUES_ROOT = HERE.parent
ACTUAL_DIR = LEAGUES_ROOT / "actual"
OUT_DIR = HERE  

LEAGUE_FILES = {
    "bundesliga": ACTUAL_DIR / "bundesliga_actual.csv",
    "la_liga": ACTUAL_DIR / "la_liga_actual.csv",
    "premier_league": ACTUAL_DIR / "premier_league_actual.csv",
    "serie_a": ACTUAL_DIR / "serie_a_actual.csv",
}

In [10]:
def load_actual(league: str) -> pd.DataFrame:
    # Load actual matches for each league and change column names
    df = pd.read_csv(LEAGUE_FILES[league])
    df = df.rename(columns={"hometeamresult": "true_home_team_result"})
    keep = ["season", "date", "home_team", "away_team", "true_home_team_result"]
    return df[keep].copy()

In [11]:
def get_home_outcome_rates(league: str) -> dict:
    """
    Find the home/draw/away probabilities for a league
    """
    adv_path = ACTUAL_DIR / "home_advantage_by_league.csv"
    df = pd.read_csv(adv_path)

    row = df.loc[df["league"].str.lower() == league.lower()]

    total = float(row["matches_total"].values[0])
    p_home = float(row["home_wins"].values[0]) / total
    p_draw = float(row["draws"].values[0]) / total
    p_away = float(row["away_wins"].values[0]) / total

    s = p_home + p_draw + p_away
    if s > 0:
        p_home, p_draw, p_away = p_home/s, p_draw/s, p_away/s

    return {"p_home": p_home, "p_draw": p_draw, "p_away": p_away}

In [12]:
def simulate_all_seeds(df_actual: pd.DataFrame, rates: dict, seeds=range(1, 11)) -> pd.DataFrame:
    """
    Simulate outcomes for each match using the probabilities from actual data.
    Here: 1 = home win, 0 = draw, -1 = away win.
    """
    out = df_actual.copy()

    # Outcomes and probabilities (ensure valid simplex)
    outcomes = [1, 0, -1]
    probs = np.array([rates["p_home"], rates["p_draw"], rates["p_away"]], dtype=float)
    probs = np.clip(probs, 0.0, 1.0)
    probs = probs / probs.sum()

    print(f"Using probabilities → Home={probs[0]:.3f}  Draw={probs[1]:.3f}  Away={probs[2]:.3f}")

    # Simulate across seeds
    for seed in seeds:
        rng = np.random.default_rng(seed)
        sim = rng.choice(outcomes, size=len(out), p=probs)
        out[f"simulated_home_team_result_seed_{seed}"] = sim

    return out


In [13]:
written = []
for league in LEAGUE_FILES.keys():
    actual_df = load_actual(league)
    rates = get_home_outcome_rates(league)
    sim_all = simulate_all_seeds(actual_df, rates, seeds=range(1, 11))

    out_path = OUT_DIR / f"{league}_simulated_matches_all_seeds.csv"
    sim_all.to_csv(out_path, index=False)
    written.append({"league": league, **rates, "rows": len(sim_all), "output": out_path.name})

pd.DataFrame(written)

Using probabilities → Home=0.449  Draw=0.249  Away=0.302
Using probabilities → Home=0.470  Draw=0.252  Away=0.279
Using probabilities → Home=0.458  Draw=0.243  Away=0.300
Using probabilities → Home=0.447  Draw=0.267  Away=0.285


,league,p_home,p_draw,p_away,rows,output
0,bundesliga,0.448646,0.249144,0.302210,6426,bundesliga_simulated_matches_all_seeds.csv
1,la_liga,0.469680,0.251602,0.278719,8740,la_liga_simulated_matches_all_seeds.csv
2,premier_league,0.457656,0.242584,0.299761,8360,premier_league_simulated_matches_all_seeds.csv
3,serie_a,0.447405,0.267434,0.285161,8518,serie_a_simulated_matches_all_seeds.csv
